# 06_D CHARMM-GUI-Style Equilibration

This notebook documents the advanced staged GROMACS equilibration workflow kept in `gromacs/charmm_gui_membrane/`.

It is not the default route for rapid polymer benchmarking. It is a conservative template for membrane proteins, enzyme/polymer systems, polymer/protein complexes, aggregation systems, and other sensitive heterogeneous systems.


## Workflow Comparison: Simplified Polymer vs CHARMM-GUI-Style

This project keeps two GROMACS equilibration workflows side by side. They are not interchangeable defaults; they serve different scientific and engineering purposes.

| Workflow | Default location | Stages | Best use | Main advantage | Main tradeoff |
| --- | --- | --- | --- | --- | --- |
| Simplified polymer workflow | `gromacs/solvated_polymer/` | minimisation -> NVT -> NPT -> production | polymer benchmarking, rapid iteration, small PHA oligomer tests, method development | fast to understand, fast to run, fewer moving parts | less conservative for fragile heterogeneous systems |
| CHARMM-GUI-style workflow | `gromacs/charmm_gui_membrane/` | minimisation -> multiple restrained/relaxation equilibration stages -> production | membrane proteins, enzyme/polymer complexes, large assemblies, sensitive interfaces | gradual relaxation reduces shock to complex systems | longer setup, more stages to inspect, more runtime |

For polymer-only benchmarking, the simplified workflow is the default because a small solvated PHA oligomer does not usually need a long staged relaxation ladder. For membrane proteins, enzyme/polymer systems, aggregation systems, or any system where interfaces and packing are delicate, the CHARMM-GUI-style staged workflow is the safer template.

Both workflows remain in the project. The folder separation is intentional so running or editing one route does not overwrite the other.


## Why the CHARMM-GUI-Style Workflow Has More Equilibration Stages

CHARMM-GUI-style protocols are deliberately conservative. They were designed for systems where abrupt relaxation can distort a membrane, damage protein interfaces, disrupt ligand/enzyme contacts, or create unstable pressure coupling early in the run.

The staged ladder typically starts with short NVT thermalisation, then several NPT relaxation stages. Early NPT stages use gentler pressure coupling and shorter runs; later stages move toward production-like pressure coupling. This gives the solvent, membrane, protein, polymer, and ions time to relax together rather than forcing the whole system directly into production conditions.

| Stage | What it does | Simulation length | Scientific reason |
| --- | --- | ---: | --- |
| `step6.0_minimization` | Energy minimisation | Not time-based | Removes steric clashes before dynamics |
| `step6.1_equilibration` | NVT thermalisation | 50 ps | Brings temperature to target while keeping the box fixed |
| `step6.2_equilibration` | Early NPT relaxation | 50 ps | Starts pressure/density relaxation gently |
| `step6.3_equilibration` | Continued NPT relaxation | 100 ps | Allows solvent and large assemblies to adjust |
| `step6.4_equilibration` | Further NPT relaxation | 100 ps | Stabilises density and interfaces |
| `step6.5_equilibration` | Production-like NPT | 200 ps | Switches toward final pressure-coupling behavior |
| `step6.6_equilibration` | Final pre-production NPT | 200 ps | Checks stability before long production |
| `step7_production` | Production MD | 100 ns by default | Generates analysis trajectory |

Use this workflow for membrane proteins, enzyme systems, polymer/protein complexes, aggregation systems, and any system where slow relaxation is more important than rapid turnaround.


## When to Use This Workflow

Use the CHARMM-GUI-style workflow when the system needs gradual relaxation:

- membrane systems where area, thickness, and water defects need careful equilibration
- enzyme/polymer systems where the binding interface should not be shocked by immediate production-like dynamics
- protein/polymer complexes where the protein fold and polymer contacts both matter
- aggregation systems where early density relaxation can change interpretation
- any system where restraints or staged release may be introduced later

Use the simplified polymer workflow instead when the system is a polymer-only solvated box and the goal is benchmarking, fast iteration, or validating the toolchain.


## Output Separation

The project keeps the two GROMACS workflows in separate folders:

| Folder | Workflow | Intended role |
| --- | --- | --- |
| `gromacs/solvated_polymer/` | simplified polymer | default polymer benchmarking and rapid iteration |
| `gromacs/charmm_gui_membrane/` | CHARMM-GUI-style staged workflow | advanced template for membrane proteins, enzyme systems, and sensitive complexes |

Do not merge these folders. The separation prevents one workflow from overwriting the other and makes it clear which protocol produced a trajectory.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd() if (Path.cwd() / "src" / "iphasimulator").exists() else Path.cwd().parent
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

system_name = "PHB4"
md_root = repo_root / "examples" / "output" / "md_tests" / system_name
gromacs_dir = md_root / "gromacs"
charmm_dir = gromacs_dir / "charmm_gui_membrane"
polymer_dir = gromacs_dir / "solvated_polymer"

{
    "repo_root": repo_root,
    "system": system_name,
    "simplified_polymer_dir": polymer_dir,
    "charmm_gui_style_dir": charmm_dir,
    "charmm_dir_exists": charmm_dir.exists(),
}


## Inspect the Generated Stage Plan

The stage descriptions below come from the reusable GROMACS runner configuration. They should match the generated SLURM script and `.mdp` templates.


In [ ]:
from iphasimulator.simulation.gromacs_runner import GROMACS_WORKFLOW_HPC_STEPS

for step_name, input_structure, description in GROMACS_WORKFLOW_HPC_STEPS["charmm_gui_membrane"]:
    print(f"{step_name:24s} input={input_structure:28s} {description}")


## Check Template Files

The CHARMM-GUI-style folder is seeded when the GROMACS run folder is prepared. If files are missing, rerun notebook `06_B_gromacs_dry_polymer.ipynb` after the GAFF2 files exist. The older `06B_gromacs_dry_polymer.ipynb` compatibility notebook performs the same folder-seeding step.


In [ ]:
required = [
    "step5_input.gro",
    "topol.top",
    "index.ndx",
    "step6.0_minimization.mdp",
    "step6.1_equilibration.mdp",
    "step6.2_equilibration.mdp",
    "step6.3_equilibration.mdp",
    "step6.4_equilibration.mdp",
    "step6.5_equilibration.mdp",
    "step6.6_equilibration.mdp",
    "step7_production.mdp",
    "run_hpc_charmm_gui_membrane.slurm",
]

for name in required:
    path = charmm_dir / name
    print(f"{name:36s} {path.exists()}  {path}")


## Execution Policy

This notebook documents and inspects the advanced protocol. It does not submit jobs automatically.

For real runs, inspect the generated `.mdp` files, confirm the structure/topology/index groups are appropriate for the biological system, then submit from notebook 07 or from the command line on the HPC system.


## Advantages and Disadvantages

| Workflow | Advantages | Disadvantages |
| --- | --- | --- |
| Simplified polymer | fewer stages, easier debugging, shorter equilibration, good default for polymer-only benchmarking | less cautious for membranes, folded proteins, or fragile complexes |
| CHARMM-GUI-style | gradual relaxation, better suited to heterogeneous or sensitive systems, closer to established biomolecular setup practice | more files, more decisions, longer wall-clock time, easier to misread if used without understanding each stage |

A good rule is to start simple for polymer-only tests, then move to the staged workflow when the system contains structured biomolecules, membranes, multiple phases, or interfaces that need careful relaxation.
